# Interpreting Logistic Regression Coefficients: Complete Guide

## 1. Title and Overview

In a logistic regression model, the estimated coefficients (beta) do NOT represent the change in the probability of the outcome for a unit change in the predictor. Instead, they represent the change in the LOG-ODDS of the outcome. To communicate model results effectively, coefficients must be transformed into Odds Ratios (OR) or Average Marginal Effects (AME).

This notebook provides a rigorous technical analysis of coefficient interpretation in logistic regression (logit) models. It details the mathematical transition from linear probability to log-odds, derives the formulas for odds ratios and marginal effects, and establishes the standard engineering practices for translating model parameters into actionable business insights.

In [ ]:
# Always start with imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from scipy import stats
import warnings

# Set display options
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)

# Set visualization style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("colorblind")

# Ignore harmless warnings for clean output
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)
print("Environment initialized. Libraries loaded.")

## 2. Setup and Data Creation: Synthetic Loan Default Dataset

We will begin by creating a synthetic dataset representing a bank's loan portfolio. The goal is to predict loan default (1 = Default, 0 = Paid) based on Age, Credit Score, and Number of Late Payments.

We simulate this strictly within the notebook using an underlying log-odds formulation to ensure we know the "true" data-generating process.

In [ ]:
def generate_loan_data(n_samples=2500):
    """Generates synthetic loan default data."""
    # Generate independent features
    age = np.random.normal(45, 10, n_samples)
    credit_score = np.random.normal(680, 60, n_samples)
    # Poisson distribution for count data
    late_payments = np.random.poisson(1.2, n_samples)
    
    # Define true log-odds generating function
    # Baseline log-odds is 10.5 (Intercept)
    # Age decreases default risk
    # Credit score strongly decreases default risk
    # Late payments heavily increase default risk
    true_log_odds = 5.0 - 0.02 * age - 0.01 * credit_score + 0.8 * late_payments
    
    # Convert log-odds to probabilities using the Sigmoid function
    probabilities = 1 / (1 + np.exp(-true_log_odds))
    
    # Generate binary outcomes based on probabilities
    default = np.random.binomial(1, probabilities)
    
    df = pd.DataFrame({
        'default': default,
        'age': np.round(age, 1),
        'credit_score': np.round(credit_score, 0),
        'late_payments': late_payments
    })
    return df

# Generate the dataset
df_loans = generate_loan_data()
print("Synthetic Loan Data Generated.")
print(f"Dataset shape: {df_loans.shape}")

In [ ]:
# Preview the data
print("\n--- First 5 rows ---")
print(df_loans.head())

print("\n--- Descriptive Statistics ---")
print(df_loans.describe().round(2))

default_rate = df_loans['default'].mean() * 100
print(f"\nOverall Default Rate: {default_rate:.2f}%")

## 3. Core Concept 1: The Log-Odds Space

In logistic regression, the relationship between the predictors X and the probability P(Y=1|X) is non-linear. Because probability is strictly bounded between 0 and 1, the effect of a predictor cannot be constant.

The model posits that the log-odds (logit) of the probability is a linear combination of the predictors:
ln(p / (1 - p)) = beta_0 + beta_1 * X_1 + ... + beta_k * X_k

Let's fit a logistic regression model using statsmodels and examine these raw coefficients.

In [ ]:
# Define features (X) and target (y)
features = ['age', 'credit_score', 'late_payments']
X = sm.add_constant(df_loans[features]) # Adds the intercept term (beta_0)
y = df_loans['default']

# Fit the Logistic Regression Model using Maximum Likelihood Estimation
logit_model = sm.Logit(y, X).fit(disp=0) # disp=0 suppresses optimizer output

print(logit_model.summary())

# Extract raw coefficients
raw_coefs = logit_model.params
print("\n--- Raw Coefficients (Log-Odds) ---")
print(raw_coefs.round(4))

### Interpreting the Log-Odds

Look at the coefficient for `late_payments` (approx +0.77). 
Interpretation: "For every additional late payment, the log-odds of loan default increase by 0.77, holding all other variables constant."

While mathematically precise, "log-odds" is highly unintuitive for business stakeholders. We must transform these into Odds Ratios.

## 4. Core Concept 2: The Odds Ratio (OR)

By exponentiating both sides of the logit equation, we isolate the odds:
Odds = p / (1 - p) = exp(beta_0 + beta_1 * X_1 + ...)

If we increase a continuous predictor X_j by one unit, the new odds multiply the old odds by exp(beta_j). 
Therefore, exp(beta_j) is the Odds Ratio. It represents the multiplicative factor by which the odds of the outcome change for a one-unit increase in X_j.

In [ ]:
# Calculate Odds Ratios and 95% Confidence Intervals
# OR = exp(beta)
odds_ratios = np.exp(raw_coefs)

# Confidence intervals for OR are derived by exponentiating the CI of the log-odds
conf_int = logit_model.conf_int()
or_conf_int = np.exp(conf_int)

# Combine into a clean DataFrame
or_df = pd.DataFrame({
    'Log-Odds (Beta)': raw_coefs,
    'Odds Ratio': odds_ratios,
    'CI_Lower_95%': or_conf_int[0],
    'CI_Upper_95%': or_conf_int[1]
})

print("--- Odds Ratios Interpretation Table ---")
print(or_df.round(4))

late_pymt_or = or_df.loc['late_payments', 'Odds Ratio']
print(f"\nBusiness Interpretation for Late Payments:")
print(f"An Odds Ratio of {late_pymt_or:.2f} means that each additional late payment multiplies the odds of default by {late_pymt_or:.2f}.")
print(f"Equivalently, it increases the odds of default by {((late_pymt_or - 1) * 100):.1f}%.")

### Visualizing Odds Ratios (Forest Plot)

A standard way to present Odds Ratios is a Forest Plot. The vertical line at OR = 1.0 represents "no effect". Variables to the right increase the odds of the event; variables to the left decrease the odds.

In [ ]:
# Remove intercept for visualization as it's just the baseline
plot_df = or_df.drop('const')
variables = plot_df.index
or_values = plot_df['Odds Ratio']
lower_errors = or_values - plot_df['CI_Lower_95%']
upper_errors = plot_df['CI_Upper_95%'] - or_values

plt.figure(figsize=(10, 5))
plt.errorbar(x=or_values, y=variables, xerr=[lower_errors, upper_errors], 
             fmt='o', color='darkred', markersize=10, capsize=5, linewidth=2)

plt.axvline(x=1.0, color='gray', linestyle='--', linewidth=2, alpha=0.7)
plt.title('Odds Ratios with 95% Confidence Intervals', fontsize=14)
plt.xlabel('Odds Ratio (exp(Beta))\n<-- Decreases Risk | Increases Risk -->', fontsize=12)
plt.ylabel('Predictor Variables', fontsize=12)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Core Concept 3: Marginal Effects

While Odds Ratios are great for understanding relative multipliers, humans intuitively think in probabilities (e.g., "What is the percentage point increase in risk?").

To find the actual change in probability, we take the partial derivative of p with respect to X_j:
dp/dX_j = beta_j * p * (1 - p)

Notice that the effect depends on the probability 'p' itself. The marginal effect is maximized when p = 0.5 and approaches zero when p is near 0 or 1. Let's visualize this mathematically.

In [ ]:
# Demonstrate how marginal effect changes based on baseline probability
beta_late = raw_coefs['late_payments']

baseline_probs = np.array([0.05, 0.50, 0.95])
marginal_effects = beta_late * baseline_probs * (1 - baseline_probs)

print(f"Coefficient for late_payments (beta): {beta_late:.4f}")
print("\nMarginal Effect (dp/dX) at different baseline probabilities:")
for p, me in zip(baseline_probs, marginal_effects):
    print(f"If customer has baseline p={p:.2f}, one more late payment adds {me*100:.1f}% to their probability.")
    
print("\nThis proves that a single coefficient translates to wildly different probability impacts depending on where you are on the curve!")

### Average Marginal Effect (AME) vs Marginal Effect at the Mean (MEM)

Because the marginal effect varies for every observation in the dataset, there are two standard approaches:
1. Marginal Effect at the Mean (MEM): Evaluate the marginal effect at the mean values of all covariates.
2. Average Marginal Effect (AME): Calculate the marginal effect for every individual observation, then take the arithmetic mean. 

AME is the industry standard because it evaluates effects on actual data points, not a hypothetical "average" person.

In [ ]:
# Calculate Average Marginal Effects (AME) using statsmodels
# method='dydx' calculates the derivative, at='overall' averages them over all observations
margeff = logit_model.get_margeff(at='overall', method='dydx')
print(margeff.summary())

# Extract values for easier interpretation
ame_values = margeff.margeff
ame_names = features

print("\n--- Additive Probability Interpretations (AME) ---")
for name, ame in zip(ame_names, ame_values):
    print(f"{name}: A 1-unit increase changes the probability of default by {(ame*100):.2f} percentage points on average.")

## 6. Visualizing the Transformation Pipeline

Let's visualize the three spaces: Log-Odds (Linear), Odds (Exponential), and Probability (Sigmoid). This makes the math completely visual.

In [ ]:
# Create synthetic range for a single predictor (Late Payments)
x_vals = np.linspace(-2, 10, 200)

# Assume baseline intercept log-odds of -4.0 and beta of 0.8
b0, b1 = -4.0, 0.8

log_odds_space = b0 + b1 * x_vals
odds_space = np.exp(log_odds_space)
prob_space = 1 / (1 + np.exp(-log_odds_space))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Log-Odds Space (Linear)
axes[0].plot(x_vals, log_odds_space, color='blue', linewidth=3)
axes[0].set_title('1. Log-Odds Space (Linear)', fontsize=14)
axes[0].set_xlabel('Predictor X (Late Payments)')
axes[0].set_ylabel('Log-Odds')
axes[0].grid(True, alpha=0.3)

# Plot 2: Odds Space (Exponential)
axes[1].plot(x_vals, odds_space, color='green', linewidth=3)
axes[1].set_title('2. Odds Space (Exponential)', fontsize=14)
axes[1].set_xlabel('Predictor X (Late Payments)')
axes[1].set_ylabel('Odds')
axes[1].grid(True, alpha=0.3)

# Plot 3: Probability Space (Non-Linear Sigmoid)
axes[2].plot(x_vals, prob_space, color='red', linewidth=3)
axes[2].set_title('3. Probability Space (Non-Linear)', fontsize=14)
axes[2].set_xlabel('Predictor X (Late Payments)')
axes[2].set_ylabel('Probability P(Y=1|X)')
axes[2].axhline(0.5, color='gray', linestyle='--', alpha=0.5)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Practical Example: A/B Testing & Conversion Rate

Let's apply this to a real-world scenario. An e-commerce platform tests a new checkout UI. The treatment group receives the new UI (1), control gets old (0). We control for user age and previous purchases.

In [ ]:
# Simulate A/B Test Data
n_ab = 3000
treatment = np.random.binomial(1, 0.5, n_ab)
user_age = np.random.normal(35, 12, n_ab)
prev_purchases = np.random.poisson(2, n_ab)

# Log-odds generating process. Treatment beta = 0.223 (approx 1.25 Odds Ratio)
lo_ab = -2.5 + 0.223 * treatment + 0.01 * user_age + 0.3 * prev_purchases
converted = np.random.binomial(1, 1 / (1 + np.exp(-lo_ab)))

df_ab = pd.DataFrame({'converted': converted, 'treatment': treatment, 
                      'user_age': user_age, 'prev_purchases': prev_purchases})

# Fit Model
X_ab = sm.add_constant(df_ab[['treatment', 'user_age', 'prev_purchases']])
y_ab = df_ab['converted']
ab_model = sm.Logit(y_ab, X_ab).fit(disp=0)

# Extract metrics for treatment variable
trt_beta = ab_model.params['treatment']
trt_or = np.exp(trt_beta)

print("\n--- A/B Test Engineering Report ---")
print(f"Log-Odds Coefficient: {trt_beta:.4f}")
print(f"Odds Ratio: {trt_or:.4f}")
print(f"\nBusiness Interpretation: Users exposed to the new checkout UI have {(trt_or - 1)*100:.1f}% higher odds of converting compared to the control group, holding age and history constant.")

## 8. Common Pitfalls: The Linear Probability Fallacy

Trap 1: Interpreting beta as a percentage point change in probability. Stating "a one-unit increase in X increases the probability of Y by beta" is mathematically false. This is the Linear Probability Model fallacy.

Let's compare the incorrect OLS coefficients to the true Logit Average Marginal Effects to see how different they are.

In [ ]:
# Fit OLS (Linear Probability Model) for comparison
ols_model = sm.OLS(y, X).fit()
ols_coefs = ols_model.params

comparison_df = pd.DataFrame({
    'Incorrect OLS Coefficient': ols_coefs,
    'Correct Logit AME (dp/dX)': [float('nan')] + list(margeff.margeff) # NaN for intercept
})

print("--- Comparing OLS Coefficients to Logit Marginal Effects ---")
print(comparison_df.dropna().round(4))
print("\nNotice that while OLS coefficients are close to the AME, using OLS directly implies the effect is constant forever, which allows probabilities > 1 and < 0. Logit AME properly summarizes the curved space.")

## 9. Practice Exercise

Scenario: A public health model assesses the probability of developing a disease based on BMI and Smoker status (1=Yes, 0=No).

Your task:
1. Fit a logistic regression model on the generated dataset.
2. Calculate the Odds Ratio for being a Smoker.
3. Calculate the Average Marginal Effect for BMI.

In [ ]:
# --- Setup Exercise Data ---
n_ex = 1500
bmi = np.random.normal(28, 5, n_ex)
smoker = np.random.binomial(1, 0.25, n_ex)
ex_log_odds = -6.0 + 0.15 * bmi + 1.386 * smoker
disease = np.random.binomial(1, 1 / (1 + np.exp(-ex_log_odds)))

df_ex = pd.DataFrame({'disease': disease, 'bmi': bmi, 'smoker': smoker})
print("Exercise data generated. Target: 'disease'. Features: 'bmi', 'smoker'")

In [ ]:
# --- Solution Block ---
X_ex = sm.add_constant(df_ex[['bmi', 'smoker']])
y_ex = df_ex['disease']

# 1. Fit the model
ex_model = sm.Logit(y_ex, X_ex).fit(disp=0)

# 2. Calculate Odds Ratio for Smoker
smoker_beta = ex_model.params['smoker']
smoker_or = np.exp(smoker_beta)
print(f"\nSolution 2: The Odds Ratio for smoking is {smoker_or:.2f}.")
print(f"Interpretation: Smokers have {smoker_or:.2f} times the odds of developing the disease compared to non-smokers.")

# 3. Calculate Average Marginal Effect for BMI
ex_margeff = ex_model.get_margeff(at='overall', method='dydx')
# The order is bmi, smoker
bmi_ame = ex_margeff.margeff[0]
print(f"\nSolution 3: The AME for BMI is {bmi_ame:.4f}.")
print(f"Interpretation: A 1-unit increase in BMI increases the absolute probability of the disease by {(bmi_ame*100):.2f} percentage points on average.")

## 10. Visualization Gallery

To fully understand interaction in a multi-variable logistic model, we can plot a 2D contour map. This shows how probability changes jointly across two continuous variables (Age and Credit Score) from our original loan dataset.

In [ ]:
# Generate grid for Age and Credit Score
age_grid = np.linspace(df_loans['age'].min(), df_loans['age'].max(), 100)
credit_grid = np.linspace(df_loans['credit_score'].min(), df_loans['credit_score'].max(), 100)
xx, yy = np.meshgrid(age_grid, credit_grid)

# Hold late_payments constant at 0
grid_const = np.ones_like(xx.ravel())
grid_late = np.zeros_like(xx.ravel())

# Predict probabilities over the grid
# Order matches the original X matrix: const, age, credit_score, late_payments
grid_X = np.column_stack((grid_const, xx.ravel(), yy.ravel(), grid_late))
Z = logit_model.predict(grid_X).reshape(xx.shape)

plt.figure(figsize=(10, 8))
contour = plt.contourf(xx, yy, Z, levels=20, cmap='RdYlGn_r', alpha=0.8)
plt.colorbar(contour, label='Probability of Default P(Y=1|X)')

plt.title('Probability Contour: Age vs Credit Score\n(Holding Late Payments = 0)', fontsize=15)
plt.xlabel('Age', fontsize=12)
plt.ylabel('Credit Score', fontsize=12)
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## 11. Summary and Key Takeaways

*   **Log-Odds (The Raw Beta)**: Logistic regression coefficients represent the change in log-odds. They are strictly additive and linear, but highly unintuitive for human interpretation.
*   **Odds Ratios (exp(Beta))**: Provide a multiplicative interpretation. A 1-unit increase in X multiplies the odds by exp(Beta). This is the standard in epidemiological and business reporting.
*   **Marginal Effects (dp/dx)**: Provide an additive, probability-based interpretation. Because the slope of the sigmoid curve changes, marginal effects depend on where you are on the curve. 
*   **Average Marginal Effect (AME)**: The industry standard for computing the overall expected impact of a variable on the actual probability of the outcome. Computed by averaging the marginal effect across all actual observations.